# 1. Analyze data
Dataset:
- Image size: 64 × 64
- Num of timesteps: 2122
- 10 actions (forward, left, back, right, jump, sneak, sprint, attack, horizontal camera change, vertical camera change)
- Actions Dimensions (2122, )
- Camera Dimensions (2122, 2)

ex:
```
$action$forward = [1, 1, 1, ..., 0, 0, 0]
$action$left = [1, 0, 1, ..., 0, 1, 0]
```

We must translate this a (2122, 10) dimension vector:
```
timestamp0 = [1, 0, 0, 0, 0, 0, 0, 0, -0.15, 0.15]
...
```


In [8]:
import numpy as np
from pathlib import Path

data_path = Path('data/MineRLTreechop-v0')

# load action samples
sample_dirs = sorted([p for p in data_path.iterdir() if p.is_dir()])

# load first sample
if sample_dirs:
    npz = np.load(sample_dirs[0] / 'rendered.npz')
    print(npz)
    # extract actions as a dictionary
    action_keys = [k for k in npz.keys() if k.startswith('action')]
    actions = {k: npz[k] for k in action_keys}

    print(f"actions: {actions}")
else:
    print(f"no actions for {sample_dirs[0]}")

NpzFile 'data/MineRLTreechop-v0/v3_absolute_grape_changeling-15_10696-12887/rendered.npz' with keys: reward, action$forward, action$left, action$back, action$right...
actions: {'action$forward': array([1, 1, 1, ..., 0, 0, 0], shape=(2122,)), 'action$left': array([0, 0, 0, ..., 0, 0, 0], shape=(2122,)), 'action$back': array([0, 0, 0, ..., 0, 0, 0], shape=(2122,)), 'action$right': array([0, 0, 0, ..., 0, 0, 0], shape=(2122,)), 'action$jump': array([0, 0, 0, ..., 0, 0, 0], shape=(2122,)), 'action$sneak': array([0, 0, 0, ..., 0, 0, 0], shape=(2122,)), 'action$sprint': array([0, 0, 0, ..., 0, 0, 0], shape=(2122,)), 'action$attack': array([0, 0, 0, ..., 1, 1, 1], shape=(2122,)), 'action$camera': array([[-0.14999998,  0.15000153],
       [-0.        ,  0.15000153],
       [ 0.        ,  0.        ],
       ...,
       [ 0.        ,  0.        ],
       [ 0.        ,  0.        ],
       [ 1.3500004 , -0.75      ]], shape=(2122, 2), dtype=float32)}


# 2. Preprocess for LeWM
 - Decode frames from recording
 - Action stacking: organize data as timestamp arrays of actions

 * Note the number of frames slightly mismatches with the number of timestamps.

In [ ]:
import cv2
import json
from pathlib import Path

# Helpers
def load_video_frames(video_path: str):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    frames = []
    f_times = []
    f_index = 0

    while True:
        # read each frame sequentially
        # ret is a boolean indicating if retrieval was successful
        ret, frame_bgr = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)
        cv2.imshow('frame_bgr', frame_rgb)
    
    print(f"frames: {len(frames)}")
    
# Load all samples

npz = np.load(sample_dirs[1] / 'rendered.npz')
video = load_video_frames( sample_dirs[1] / 'recording.mp4')
with open(sample_dirs[1]/'metadata.json', 'r') as f:
    metadata = json.load(f)

# Match each sample frame. To reconcile the mismatch in frames


frames: 2310
